In [1]:
# This reads your .env file and loads your secret API key into memory
# Without this, Python doesn't know your API key exists

import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
print("✅ API key loaded successfully")

✅ API key loaded successfully


In [2]:
# This is our fictional customer Sarah
# In a real bank this would come from a database
# "Mock data" = fake but realistic data for testing

customer_transactions = """
Customer ID: CUST_001
Name: Sarah Mitchell
Age: 34

Transaction History (Last 6 Months):
- Ryanair flight booking: €320 (March)
- Aer Lingus flight booking: €480 (March)
- Currency exchange - USD: €200 (March)
- Hotel booking - New York: €850 (April)
- Airport parking: €45 (April)
- Ryanair flight booking: €210 (May)
- Currency exchange - EUR to GBP: €150 (May)
- Hotel booking - London: €420 (May)
- Airport lounge fee: €35 (May)
- Ryanair flight booking: €390 (June)
- Currency exchange - EUR to JPY: €300 (June)
- Hotel booking - Tokyo: €1,200 (June)
- Travel insurance: €65 (June)
- Airport parking: €55 (July)
- Aer Lingus flight booking: €520 (July)
- Supermarket shopping: €120 (weekly average)
- Gym membership: €45/month
- Netflix subscription: €15/month
- Electricity bill: €180/month

Current Banking Products:
- Basic current account
- Standard debit card (no travel benefits)
- No credit card
- No savings account
"""

print("✅ Customer data ready")

✅ Customer data ready


In [3]:
# A prompt is the instruction we give to Claude
# Think of it as a task brief you'd give a smart analyst
# The f""" """ syntax lets us insert the customer data directly into the text

prompt = f"""
You are a Customer Intelligence Agent for a bank.

Your job is to analyse a customer's transaction history and produce a structured 
behavioural profile. You must identify patterns and translate them into clear 
affinities and needs — WITHOUT referencing specific sensitive transactions.

Here is the customer data:

{customer_transactions}

Produce a Customer Intelligence Profile in exactly this format:

CUSTOMER INTELLIGENCE PROFILE
==============================
Customer ID: [id]
Name: [name]

BEHAVIOURAL AFFINITIES:
- Travel Affinity: [LOW / MEDIUM / HIGH] — [one sentence reason]
- FX/International Need: [LOW / MEDIUM / HIGH] — [one sentence reason]  
- Premium Product Fit: [LOW / MEDIUM / HIGH] — [one sentence reason]
- Savings Propensity: [LOW / MEDIUM / HIGH] — [one sentence reason]

TOP IDENTIFIED NEED:
[One sentence describing the single most important unmet need]

RECOMMENDED CONVERSATION TRIGGER:
[One sentence on what the executive could mention, without revealing specific transactions]

PRIVACY CHECK:
[Confirm that no specific sensitive transaction details appear in this profile]
"""

print("✅ Prompt ready")

✅ Prompt ready


In [4]:
client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-haiku-4-5",   
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(message.content[0].text)

CUSTOMER INTELLIGENCE PROFILE
Customer ID: CUST_001
Name: Sarah Mitchell

BEHAVIOURAL AFFINITIES:
- Travel Affinity: HIGH — Customer has booked multiple international flights and accommodations across three continents within a six-month period, indicating frequent leisure or business travel.
- FX/International Need: HIGH — Repeated currency exchanges across USD, GBP, and JPY demonstrate active international spending and exposure to multiple currency zones.
- Premium Product Fit: HIGH — Travel frequency, international transactions, and airport service usage (lounge access, parking) suggest alignment with premium travel-focused banking products.
- Savings Propensity: MEDIUM — Regular subscriptions and consistent essential spending indicate financial discipline, though no dedicated savings vehicle is currently utilized.

TOP IDENTIFIED NEED:
Sarah requires a travel-optimized payment solution that eliminates currency exchange friction and maximizes rewards on international spending, which 

In [5]:
# chromadb is our vector database — stores meaning, not just words
# sentence_transformers converts text into vectors (lists of numbers representing meaning)

import chromadb
from chromadb.utils import embedding_functions
import json

# This is the "embedding model" — the thing that converts text into meaning-numbers
# It runs locally on your laptop, no API key needed, completely free
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"  # a small, fast, free model perfect for this
)

# Create the vector database in memory (disappears when notebook closes — fine for now)
client = chromadb.Client()

# Create a "collection" — think of it like a table in a normal database
# but instead of rows, it stores meaning-vectors
collection = client.create_collection(
    name="bank_products",
    embedding_function=embedding_fn
)

print("✅ Vector database ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector database ready


In [6]:
# These are our mock bank products
# In a real system, these would be scraped from the bank's website automatically
# For now we're defining them manually — the "structured data" foundation we discussed

products = [
    {
        "id": "PROD_001",
        "name": "Voyager Premium Travel Card",
        "description": """Premium credit card designed for frequent international travellers. 
        Includes unlimited airport lounge access worldwide, zero foreign transaction fees, 
        travel insurance, and 3x reward points on flights and hotels. 
        Annual fee: €150. Requires minimum income of €35,000."""
    },
    {
        "id": "PROD_002", 
        "name": "GlobalCash FX Account",
        "description": """Multi-currency account that holds and converts between 30 currencies 
        at interbank rates with no markup. Ideal for customers who regularly send money abroad 
        or make international purchases. No monthly fee. Includes a linked debit card."""
    },
    {
        "id": "PROD_003",
        "name": "HomeSaver Account",
        "description": """High-interest savings account for customers saving towards a home purchase. 
        Offers 4.5% AER on balances up to €50,000. Requires minimum monthly deposit of €200. 
        Bonus interest rate if no withdrawals made in 12 months."""
    },
    {
        "id": "PROD_004",
        "name": "FlexiLoan Personal Loan",
        "description": """Unsecured personal loan from €1,000 to €50,000 with flexible repayment 
        terms of 1 to 7 years. Competitive fixed interest rate. Suitable for home improvements, 
        car purchase, or consolidating existing debts. Fast online approval."""
    },
    {
        "id": "PROD_005",
        "name": "FamilyFirst Current Account",
        "description": """Family banking package that includes accounts for parents and up to 3 children. 
        Includes pocket money features, spending controls for children, joint account option, 
        and family cashback on supermarket and school-related spending. Monthly fee: €8."""
    }
]

print(f"✅ {len(products)} products defined")
for p in products:
    print(f"   - {p['id']}: {p['name']}")

✅ 5 products defined
   - PROD_001: Voyager Premium Travel Card
   - PROD_002: GlobalCash FX Account
   - PROD_003: HomeSaver Account
   - PROD_004: FlexiLoan Personal Loan
   - PROD_005: FamilyFirst Current Account


In [7]:
# This is the key step — we're converting each product description into a vector
# and storing it in the database
# chromadb does this automatically using the embedding function we set up in Cell 1

collection.add(
    ids=[p["id"] for p in products],           # unique ID for each product
    documents=[p["description"] for p in products],  # the text that gets vectorised
    metadatas=[{"name": p["name"], "id": p["id"]} for p in products]  # extra info to retrieve
)

print("✅ All products loaded into vector database")
print(f"   Total products stored: {collection.count()}")

✅ All products loaded into vector database
   Total products stored: 5


In [8]:
# This is the magic moment
# We're searching by MEANING, not by keyword
# Watch what happens when we search for customer needs

# Query 1: Travel customer
query = "frequent international traveller who pays foreign exchange fees and uses airport lounges"

results = collection.query(
    query_texts=[query],   # natural language — no keywords needed
    n_results=2            # return top 2 most relevant products
)

print("🔍 QUERY:", query)
print("=" * 60)
print("TOP MATCHING PRODUCTS:\n")

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"Match #{i+1}: {meta['name']} ({meta['id']})")
    print(f"Description: {doc[:200]}...")  # first 200 characters
    print()

🔍 QUERY: frequent international traveller who pays foreign exchange fees and uses airport lounges
TOP MATCHING PRODUCTS:

Match #1: Voyager Premium Travel Card (PROD_001)
Description: Premium credit card designed for frequent international travellers. 
        Includes unlimited airport lounge access worldwide, zero foreign transaction fees, 
        travel insurance, and 3x reward...

Match #2: GlobalCash FX Account (PROD_002)
Description: Multi-currency account that holds and converts between 30 currencies 
        at interbank rates with no markup. Ideal for customers who regularly send money abroad 
        or make international purc...



In [9]:
# Now try a completely different customer profile
# This shows how the same database serves different needs

query2 = "customer who wants to save money and is not spending wisely"

results2 = collection.query(
    query_texts=[query2],
    n_results=2
)

print("🔍 QUERY:", query2)
print("=" * 60)
print("TOP MATCHING PRODUCTS:\n")

for i, (doc, meta) in enumerate(zip(results2['documents'][0], results2['metadatas'][0])):
    print(f"Match #{i+1}: {meta['name']} ({meta['id']})")
    print(f"Description: {doc[:200]}...")
    print()

🔍 QUERY: customer who wants to save money and is not spending wisely
TOP MATCHING PRODUCTS:

Match #1: HomeSaver Account (PROD_003)
Description: High-interest savings account for customers saving towards a home purchase. 
        Offers 4.5% AER on balances up to €50,000. Requires minimum monthly deposit of €200. 
        Bonus interest rate i...

Match #2: FamilyFirst Current Account (PROD_005)
Description: Family banking package that includes accounts for parents and up to 3 children. 
        Includes pocket money features, spending controls for children, joint account option, 
        and family cashb...

